# SpecRAG: GPU-Accelerated Public Multi-Benchmark Harness

> **Run large-scale RAG evaluations on a free Google Colab T4 GPU.**
> Evaluates Naive Baseline vs. Production Hybrid RAG across 4 official third-party benchmarks:
- **1. T²-RAGBench (`FinQA`)**: `G4KMU/t2-ragbench` — SEC 10-K filings, numerical tables & financial balance sheets.
- **2. TAT-DQA (`TAT-QA`)**: `G4KMU/t2-ragbench` — Tabular & textual document QA with complex multi-row financial grids.
- **3. IBM TechReport**: `ibm-research/REAL-MM-RAG_TechReport` — Microarchitectures, bus links, and hexadecimal register specs.
- **4. TechQA**: `galileo-ai/ragbench` — Enterprise technical support & engineering specifications.

Supports both **Fast Offline Verification** (10-second instant check) and **Live Hugging Face Streaming** (hundreds of real enterprise documents & distractor tables indexed into Qdrant).

In [ ]:
# 1. Verify NVIDIA GPU Acceleration
!nvidia-smi

In [ ]:
# 2. Clone or Update Repository and Install Dependencies
import os
if not os.path.exists("/content/specrag"):
    !git clone https://github.com/frasheed-dev/specrag.git /content/specrag
%cd /content/specrag
!git pull origin main

!pip install -q uv
!uv pip install -q -e ".[dev]"

In [ ]:
# 3. (Optional) Set Hugging Face Token to avoid Hub rate limits
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass

In [ ]:
# ==============================================================================
# 4A. Fast Offline Benchmark (Deterministic Ground-Truth Packs)
# ==============================================================================
# Runs in ~10 seconds with zero internet download required.
# Evaluates 40 verified golden queries across financial tables and hardware specs.
#
# Options:
#   --dataset (-d) : 'all', 't2' (Financial Tables), 'ibm' (Processor Specs)
#   --source (-s)  : 'local' (loads pre-packaged offline JSON packs from data/)
# ==============================================================================
!python -m specrag.cli benchmark --dataset all --source local

In [ ]:
# ==============================================================================
# 4B. Full-Scale Live Hugging Face Multi-Corpus Benchmark
# ==============================================================================
# Streams real public datasets directly from Hugging Face Hub into Qdrant.
# Evaluates against genuine distractor noise (hundreds of documents & tables).
#
# Available CLI Parameters:
#   --dataset (-d) :
#       • 'all'     -> Evaluates all 4 public benchmarks in sequence
#       • 't2'      -> G4KMU/t2-ragbench (FinQA: SEC 10-K financial tables)
#       • 'tatqa'   -> G4KMU/t2-ragbench (TAT-DQA: Complex financial grids)
#       • 'ibm'     -> ibm-research/REAL-MM-RAG_TechReport (Hardware specs)
#       • 'techqa'  -> galileo-ai/ragbench (TechQA: Enterprise engineering)
#
#   --source (-s)  :
#       • 'hf'      -> Streams live dataset splits from Hugging Face Hub
#       • 'local'   -> Uses offline cached files
#
#   --limit (-n)   :
#       • 50 / 100  -> Samples N documents (e.g. 100 docs ≈ 500 chunks in Qdrant)
#       • None      -> Ingests full dataset split (unlimited)
#
# Examples to try:
#   !python -m specrag.cli benchmark --dataset all --source hf --limit 50
#   !python -m specrag.cli benchmark --dataset tatqa --source hf --limit 50
#   !python -m specrag.cli benchmark --dataset ibm --source hf --limit 50
# ==============================================================================
!python -m specrag.cli benchmark --dataset all --source hf --limit 50

In [ ]:
# 5. Display Formatted Benchmark Results Report
from IPython.display import Markdown

with open("BENCHMARK_RESULTS.md") as f:
    display(Markdown(f.read()))